# Polaris ETL Flow Test

**Purpose:** Verify full ETL read/write flow through Polaris  
**Tests:** token → read table → verify STS → write table → verify snapshot

## Test sequence
```
E01: Read source table (bronze) → verify TABLE_READ_DATA
E02: Verify STS credentials scoped correctly
E03: Write target table (silver) → verify TABLE_WRITE_DATA
E04: Verify new snapshot committed
E05: Verify RBAC blocks cross-catalog access
```

In [ ]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_ROOT = pathlib.Path.cwd()
while not (_ROOT / "src").is_dir() and _ROOT != _ROOT.parent:
    _ROOT = _ROOT.parent
_SRC = _ROOT / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
from nb_support import *
require_not_prod("etl-flow test")  # PROD-safe: refuses to run against company PROD

tr = TestResult()

# Login
token = get_token()
print(f"✅ Token acquired")

# Show available catalogs
r = list_catalogs(token)
if r.status_code == 200:
    print(f"Available catalogs: {[c['name'] for c in r.json().get('catalogs', [])]}")

In [ ]:
# E01: Read source table
print("\nE01: Read source table (bronze)")

# List namespaces in bronze
r = requests.get(
    f"{POLARIS_URL}/api/catalog/v1/namespaces",
    headers={**polaris_headers(token), "warehouse": "bronze"}
)
passed = r.status_code == 200
detail = f"HTTP {r.status_code}"
if passed:
    ns_list = r.json().get("namespaces", [])
    detail = f"namespaces: {ns_list}"
tr.record("E01_read_namespace_list", passed, detail)

In [ ]:
# E02: Verify STS scoped correctly
print("\nE02: STS credential scoping")

# List tables in first namespace
if r.status_code == 200 and r.json().get("namespaces"):
    first_ns = r.json()["namespaces"][0][0]
    r_tables = requests.get(
        f"{POLARIS_URL}/api/catalog/v1/namespaces/{first_ns}/tables",
        headers={**polaris_headers(token), "warehouse": "bronze"}
    )
    if r_tables.status_code == 200 and r_tables.json().get("identifiers"):
        first_table = r_tables.json()["identifiers"][0]["name"]

        # Get STS credentials for this table
        r_creds = get_credentials(token, "bronze", first_ns, first_table)
        passed = r_creds.status_code == 200
        if passed:
            creds = r_creds.json().get("config", {}).get("credentials", {})
            location = r_creds.json().get("config", {}).get("uri", "")
            detail = f"STS issued, location: {location}"
        else:
            detail = f"HTTP {r_creds.status_code}"
        tr.record("E02_sts_scoped", passed, detail)
    else:
        tr.record("E02_sts_scoped", False, "no tables found in bronze namespace")
else:
    tr.record("E02_sts_scoped", False, "no namespaces in bronze")

In [ ]:
# E03: Write to watchdog catalog (simulate ETL write)
print("\nE03: Write target table (watchdog catalog)")

# Setup
create_namespace(token, WATCHDOG_CATALOG, "etl-test-ns")

r_write = create_table(
    token, WATCHDOG_CATALOG, "etl-test-ns",
    "etl-output-table",
    f"{WATCHDOG_LOCATION}etl-output/"
)
passed = r_write.status_code == 200
tr.record("E03_write_table", passed, f"HTTP {r_write.status_code}")

In [ ]:
# E04: Verify table written (new snapshot)
print("\nE04: Verify table committed")
r_get = get_table(token, WATCHDOG_CATALOG, "etl-test-ns", "etl-output-table")
passed = r_get.status_code == 200
detail = f"HTTP {r_get.status_code}"
if passed:
    meta_loc = r_get.json().get("metadata-location", "")
    detail = f"metadata-location: {meta_loc[:60]}..."
tr.record("E04_snapshot_committed", passed, detail)

In [ ]:
# E05: Verify cross-catalog write blocked
print("\nE05: Cross-catalog write blocked")
# Try to write to gold catalog (watchdog should not have access)
r_blocked = create_table(
    token, "gold", "watchdog-ns",
    "unauthorized-table",
    "s3://benchmarks/gold/watchdog/"
)
passed = r_blocked.status_code in [403, 404]
tr.record("E05_cross_catalog_blocked", passed,
    f"HTTP {r_blocked.status_code} ({'blocked correctly' if passed else 'NOT BLOCKED — RBAC issue'})")

In [ ]:
# Cleanup
print("\nCleanup...")
delete_table(token, WATCHDOG_CATALOG, "etl-test-ns", "etl-output-table")
delete_namespace(token, WATCHDOG_CATALOG, "etl-test-ns")
print("  cleaned")

# Summary
print()
tr.summary()